# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/A1704/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Rule

Prioritize content for refresh review when it is both **stale and still receiving meaningful search visibility**. This is a transparent decision-support rule for reviewing content, not a prediction of future performance.

### Reason code

- `stale_but_visible` — the content has not been updated recently and has sufficient search impressions to justify review.

### Planned action

- `review_refresh` — send the item for human review before deciding whether to refresh it.

### Thresholds

- Stale: `days_since_last_update >= 180`
- Meaningful visibility: `impressions_90d >= 500`

Items that do not meet both conditions receive a score of zero and remain lower in the ranked queue.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

# Find the dataset if the notebook is running in Google Colab.
if not DATA_PATH.exists():
    candidates = [
        Path("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"),
        Path("/content/content_refresh_anonymized.csv"),
    ]
    DATA_PATH = next((p for p in candidates if p.exists()), DATA_PATH)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Dataset not found. We need to locate the repository data file first."
    )

df = pd.read_csv(DATA_PATH)

required = [
    "content_id",
    "client_id",
    "days_since_last_update",
    "impressions_90d",
    "trend_direction",
    "trend_pct",
]

missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Dataset loaded:", df.shape)
print("Unique clients:", df["client_id"].nunique())
print("Required columns verified.")


Dataset loaded: (30000, 44)
Unique clients: 32
Required columns verified.


In [15]:
!git clone https://github.com/A1704/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [16]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)

required = [
    "content_id",
    "client_id",
    "days_since_last_update",
    "impressions_90d",
    "trend_direction",
    "trend_pct",
]

missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Dataset shape:", df.shape)
print("Unique clients:", df["client_id"].nunique())
print("\nTrend direction counts:")
print(df["trend_direction"].value_counts(dropna=False))

Dataset shape: (30000, 44)
Unique clients: 32

Trend direction counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


In [17]:
# Signal-audit label, derived exactly as documented.
# Use only for checking signals, NEVER as a scoring feature.
df["audit_declining_label"] = (
    df["trend_direction"].eq("down").astype(int)
)

# Signal 1: Time since last update
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-np.inf, 90, 180, np.inf],
    labels=["0–90 days", "91–180 days", "181+ days"],
)

staleness_check = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("audit_declining_label", "size"),
          decline_rate=("audit_declining_label", "mean"),
      )
      .reset_index()
)

print("SIGNAL 1 — STALENESS")
display(staleness_check)

# Signal 2: Search visibility
df["visibility_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[-np.inf, 499, 2999, np.inf],
    labels=["Low: <500", "Moderate: 500–2999", "High: 3000+"],
)

visibility_check = (
    df.groupby("visibility_bucket", observed=False)
      .agg(
          n=("audit_declining_label", "size"),
          decline_rate=("audit_declining_label", "mean"),
      )
      .reset_index()
)

print("SIGNAL 2 — SEARCH VISIBILITY")
display(visibility_check)

SIGNAL 1 — STALENESS


,staleness_bucket,n,decline_rate
0,0–90 days,20655,0.512031
1,91–180 days,9171,0.611057
2,181+ days,174,0.471264


SIGNAL 2 — SEARCH VISIBILITY


,visibility_bucket,n,decline_rate
0,Low: <500,13274,0.474687
1,Moderate: 500–2999,8443,0.620632
2,High: 3000+,8283,0.569963


### Signal audit findings

**Signal 1 — Staleness: MIXED**

- 0–90 days: n = 20,655; decline rate = 51.2%.
- 91–180 days: n = 9,171; decline rate = 61.1%.
- 181+ days: n = 174; decline rate = 47.1%.

The decline rate is highest in the 91–180-day bucket, then falls in the 181+ bucket. The oldest bucket is small, so these results do not support a simple monotonic relationship between staleness and decline.

**Signal 2 — Search visibility: MIXED**

- Under 500 impressions: n = 13,274; decline rate = 47.5%.
- 500–2,999 impressions: n = 8,443; decline rate = 62.1%.
- 3,000+ impressions: n = 8,283; decline rate = 57.0%.

The moderate-visibility bucket has the highest observed decline rate. The decline rate falls in the high-visibility bucket, so visibility alone does not show a monotonic relationship with decline.

**Limitations:** These are descriptive results from the 30,000-row snapshot. They are not causal evidence or proof of future performance. The audit label is used only to evaluate the signals and must not be included in the baseline score.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Baseline Action Score — transparent, rule-based prioritization

from pathlib import Path
import pandas as pd
import numpy as np

# Validate the inputs needed for the scoring rule
required = [
    "content_id",
    "client_id",
    "days_since_last_update",
    "impressions_90d",
]

missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Make a separate queue so the original dataset remains unchanged
queue = df[required].copy()

# Explicit thresholds
STALE_DAYS = 180
MIN_IMPRESSIONS = 500

# One transparent score: stale AND visible
queue["is_stale"] = (
    queue["days_since_last_update"].ge(STALE_DAYS)
    & queue["days_since_last_update"].notna()
)

queue["is_visible"] = (
    queue["impressions_90d"].ge(MIN_IMPRESSIONS)
    & queue["impressions_90d"].notna()
)

queue["score"] = (
    queue["is_stale"].astype(int)
    * queue["is_visible"].astype(int)
    * queue["impressions_90d"].fillna(0)
)

# Exactly ONE reason code for every row
queue["reason_code"] = np.where(
    queue["score"] > 0,
    "stale_but_visible",
    "not_priority_by_rule",
)

queue["action"] = np.where(
    queue["score"] > 0,
    "review_refresh",
    "no_priority",
)

# Highest scores first, with deterministic tie-breakers
queue = queue.sort_values(
    ["score", "content_id"],
    ascending=[False, True],
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

# Write the requested output
output_path = Path(
    "/content/flyrank-ml-internship/work/outputs/baseline_action_score.csv"
)
output_path.parent.mkdir(parents=True, exist_ok=True)

queue.to_csv(output_path, index=False)

print("Rows ranked:", len(queue))
print("Items prioritized:", int((queue["score"] > 0).sum()))
print("CSV saved to:", output_path)
print("\nTop 10 ranked items:")
display(queue.head(10))


Rows ranked: 30000
Items prioritized: 17
CSV saved to: /content/flyrank-ml-internship/work/outputs/baseline_action_score.csv

Top 10 ranked items:


,content_id,client_id,days_since_last_update,impressions_90d,is_stale,is_visible,score,reason_code,action,rank
0,content_cf56e2e2e282,client_7f2253d7e2,194,61678,True,True,61678,stale_but_visible,review_refresh,1
1,content_7368877ea310,client_7f2253d7e2,194,59472,True,True,59472,stale_but_visible,review_refresh,2
2,content_1bfaa38ff26c,client_7f2253d7e2,194,25715,True,True,25715,stale_but_visible,review_refresh,3
3,content_0a91db491d14,client_7f2253d7e2,193,13299,True,True,13299,stale_but_visible,review_refresh,4
4,content_5feee3994adb,client_7f2253d7e2,194,7812,True,True,7812,stale_but_visible,review_refresh,5
5,content_c2d929d83eaa,client_7f2253d7e2,193,7558,True,True,7558,stale_but_visible,review_refresh,6
6,content_b16bd7307b39,client_7f2253d7e2,194,4590,True,True,4590,stale_but_visible,review_refresh,7
7,content_fe16a55cd13d,client_7f2253d7e2,194,4556,True,True,4556,stale_but_visible,review_refresh,8
8,content_ecb6215e79fd,client_7f2253d7e2,194,4429,True,True,4429,stale_but_visible,review_refresh,9
9,content_928af3e22c80,client_7f2253d7e2,193,1697,True,True,1697,stale_but_visible,review_refresh,10


In [19]:
# Baseline coverage and score distribution
print("Total items:", len(queue))
print("Prioritized items:", int((queue["score"] > 0).sum()))
print("Priority rate:", round((queue["score"] > 0).mean() * 100, 2), "%")

print("\nReason-code counts:")
display(queue["reason_code"].value_counts().rename_axis("reason_code").reset_index(name="n"))

print("\nPrioritized items by client:")
display(
    queue.loc[queue["score"] > 0]
         .groupby("client_id")
         .agg(
             n=("content_id", "size"),
             min_score=("score", "min"),
             max_score=("score", "max"),
         )
         .sort_values("n", ascending=False)
         .reset_index()
)

print("\nThreshold check:")
print("Stale items:", int(queue["is_stale"].sum()))
print("Visible items:", int(queue["is_visible"].sum()))
print(
    "Both stale and visible:",
    int((queue["is_stale"] & queue["is_visible"]).sum())
)

Total items: 30000
Prioritized items: 17
Priority rate: 0.06 %

Reason-code counts:


,reason_code,n
0,not_priority_by_rule,29983
1,stale_but_visible,17



Prioritized items by client:


,client_id,n,min_score,max_score
0,client_7f2253d7e2,12,828,61678
1,client_d029fa3a95,3,533,1408
2,client_4ec9599fc2,1,821,821
3,client_9400f1b21c,1,954,954



Threshold check:
Stale items: 174
Visible items: 16726
Both stale and visible: 17


In [20]:
top20_review = queue.head(20).copy()

top20_review["confidence_note"] = np.where(
    (top20_review["is_stale"]) & (top20_review["is_visible"]),
    "Moderate confidence in rule match; page quality not verified",
    "Low confidence; item does not meet both priority conditions"
)

top20_review["what_would_make_it_wrong"] = np.where(
    top20_review["is_stale"] & top20_review["is_visible"],
    "High impressions may not mean the page needs refreshing; verify intent, quality, and business value",
    "The item may still deserve review for reasons not captured by this rule"
)

top20_review = top20_review[
    [
        "rank",
        "content_id",
        "score",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong",
    ]
]

display(top20_review)

,rank,content_id,score,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_cf56e2e2e282,61678,review_refresh,stale_but_visible,Moderate confidence in rule match; page qualit...,High impressions may not mean the page needs r...
1,2,content_7368877ea310,59472,review_refresh,stale_but_visible,Moderate confidence in rule match; page qualit...,High impressions may not mean the page needs r...
2,3,content_1bfaa38ff26c,25715,review_refresh,stale_but_visible,Moderate confidence in rule match; page qualit...,High impressions may not mean the page needs r...
3,4,content_0a91db491d14,13299,review_refresh,stale_but_visible,Moderate confidence in rule match; page qualit...,High impressions may not mean the page needs r...
4,5,content_5feee3994adb,7812,review_refresh,stale_but_visible,Moderate confidence in rule match; page qualit...,High impressions may not mean the page needs r...
5,6,content_c2d929d83eaa,7558,review_refresh,stale_but_visible,Moderate confidence in rule match; page qualit...,High impressions may not mean the page needs r...
6,7,content_b16bd7307b39,4590,review_refresh,stale_but_visible,Moderate confidence in rule match; page qualit...,High impressions may not mean the page needs r...
7,8,content_fe16a55cd13d,4556,review_refresh,stale_but_visible,Moderate confidence in rule match; page qualit...,High impressions may not mean the page needs r...
8,9,content_ecb6215e79fd,4429,review_refresh,stale_but_visible,Moderate confidence in rule match; page qualit...,High impressions may not mean the page needs r...
9,10,content_928af3e22c80,1697,review_refresh,stale_but_visible,Moderate confidence in rule match; page qualit...,High impressions may not mean the page needs r...


In [21]:

# Weak-pick review: inspect the lowest-scoring prioritized items
weak_picks = queue.loc[queue["score"] > 0].tail(5).copy()

print("Five weakest prioritized picks:")
display(
    weak_picks[
        [
            "rank",
            "content_id",
            "days_since_last_update",
            "impressions_90d",
            "score",
            "reason_code",
            "action",
        ]
    ]
)

# Leakage check: ensure target/label fields are not used in the score
forbidden_score_fields = {
    "trend_pct",
    "trend_direction",
    "audit_declining_label",
    "is_declining_label",
}

score_inputs = {
    "days_since_last_update",
    "impressions_90d",
}

print("\nLeakage check:")
print("Score inputs:", sorted(score_inputs))
print("Forbidden label fields:", sorted(forbidden_score_fields))
print("Forbidden fields used as score inputs:",
      sorted(score_inputs & forbidden_score_fields))

assert not (score_inputs & forbidden_score_fields), \
    "Potential target leakage detected in score inputs"

print("\nRows in queue:", len(queue))
print("Unique content IDs:", queue["content_id"].nunique())
print("Queue ranks unique:", queue["rank"].is_unique)
print("One reason code per row:", queue["reason_code"].notna().all())
print("One action per row:", queue["action"].notna().all())

assert len(queue) == 30000
assert queue["content_id"].nunique() == len(queue)
assert queue["rank"].is_unique
assert queue["reason_code"].notna().all()
assert queue["action"].notna().all()


Five weakest prioritized picks:


,rank,content_id,days_since_last_update,impressions_90d,score,reason_code,action
12,13,content_7f116ae1f6f5,301,954,954,stale_but_visible,review_refresh
13,14,content_77d4d5930e5e,194,828,828,stale_but_visible,review_refresh
14,15,content_72496874f806,301,821,821,stale_but_visible,review_refresh
15,16,content_6226ee6adc91,183,545,545,stale_but_visible,review_refresh
16,17,content_074ba6ead17b,183,533,533,stale_but_visible,review_refresh



Leakage check:
Score inputs: ['days_since_last_update', 'impressions_90d']
Forbidden label fields: ['audit_declining_label', 'is_declining_label', 'trend_direction', 'trend_pct']
Forbidden fields used as score inputs: []

Rows in queue: 30000
Unique content IDs: 30000
Queue ranks unique: True
One reason code per row: True
One action per row: True


In [22]:

import json
from pathlib import Path

output_dir = Path("/content/flyrank-ml-internship/work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

prioritized = queue["score"] > 0

metrics = {
    "assignment": "ML-07 — Baseline Action Score and Top-20 Review",
    "dataset_rows": int(len(queue)),
    "unique_content_ids": int(queue["content_id"].nunique()),
    "prioritized_items": int(prioritized.sum()),
    "priority_rate_pct": round(float(prioritized.mean() * 100), 4),
    "stale_items": int(queue["is_stale"].sum()),
    "visible_items": int(queue["is_visible"].sum()),
    "stale_and_visible_items": int(
        (queue["is_stale"] & queue["is_visible"]).sum()
    ),
    "reason_code_counts": {
        str(k): int(v)
        for k, v in queue["reason_code"].value_counts().items()
    },
    "unique_ranks": bool(queue["rank"].is_unique),
    "unique_content_ids_check": bool(
        queue["content_id"].nunique() == len(queue)
    ),
    "forbidden_score_fields_used": [],
    "score_rule": (
        "days_since_last_update >= 180 AND impressions_90d >= 500; "
        "qualifying items are ranked by impressions_90d"
    ),
    "limitations": [
        "Only 17 items qualify under the chosen thresholds.",
        "Prioritized items are concentrated in a small number of clients.",
        "The score is a decision-support heuristic, not a validated predictor.",
        "Page quality and the value of refreshing each item require human review."
    ]
}

metrics_path = output_dir / "ml07_baseline_metrics.json"
with metrics_path.open("w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("Metrics saved to:", metrics_path)
print(json.dumps(metrics, indent=2))


Metrics saved to: /content/flyrank-ml-internship/work/outputs/ml07_baseline_metrics.json
{
  "assignment": "ML-07 \u2014 Baseline Action Score and Top-20 Review",
  "dataset_rows": 30000,
  "unique_content_ids": 30000,
  "prioritized_items": 17,
  "priority_rate_pct": 0.0567,
  "stale_items": 174,
  "visible_items": 16726,
  "stale_and_visible_items": 17,
  "reason_code_counts": {
    "not_priority_by_rule": 29983,
    "stale_but_visible": 17
  },
  "unique_ranks": true,
  "unique_content_ids_check": true,
  "forbidden_score_fields_used": [],
  "score_rule": "days_since_last_update >= 180 AND impressions_90d >= 500; qualifying items are ranked by impressions_90d",
  "limitations": [
    "Only 17 items qualify under the chosen thresholds.",
    "Prioritized items are concentrated in a small number of clients.",
    "The score is a decision-support heuristic, not a validated predictor.",
    "Page quality and the value of refreshing each item require human review."
  ]
}


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 review findings

The ranked queue contains 17 items that meet both rule conditions and 3 zero-score items in positions 18–20. Each qualifying item receives the `review_refresh` action and `stale_but_visible` reason code. The remaining three receive `no_priority` and `not_priority_by_rule`.

The rule-based confidence notes are provisional: they describe how closely an item matches the rule, not whether refreshing the page will improve performance. The highest-ranked items have more impressions, but impressions alone do not establish content quality, business value, or refresh potential.

The lowest-ranked qualifying items are particularly uncertain because they are only slightly above the 500-impression threshold. Human review should verify page quality, search intent, current relevance, and business value before any refresh decision.

A top-ranked item could be a false positive if it is still useful and current despite its age. A zero-score item could be a false negative if it needs attention for a reason not captured by the rule. These results are directional decision-support signals, not causal evidence or predictions of future performance.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


top20_review = queue.head(20).copy()

top20_review["confidence_note"] = np.where(
    (top20_review["is_stale"]) & (top20_review["is_visible"]),
    "Moderate confidence in rule match; page quality not verified",
    "Low confidence; item does not meet both priority conditions"
)

top20_review["what_would_make_it_wrong"] = np.where(
    top20_review["is_stale"] & top20_review["is_visible"],
    "High impressions may not mean the page needs refreshing; verify intent, quality, and business value",
    "The item may still deserve review for reasons not captured by this rule"
)

top20_review = top20_review[
    [
        "rank", "content_id", "score", "action",
        "reason_code", "confidence_note", "what_would_make_it_wrong"
    ]
]

display(top20_review)



,rank,content_id,score,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_cf56e2e2e282,61678,review_refresh,stale_but_visible,Moderate confidence in rule match; page qualit...,High impressions may not mean the page needs r...
1,2,content_7368877ea310,59472,review_refresh,stale_but_visible,Moderate confidence in rule match; page qualit...,High impressions may not mean the page needs r...
2,3,content_1bfaa38ff26c,25715,review_refresh,stale_but_visible,Moderate confidence in rule match; page qualit...,High impressions may not mean the page needs r...
3,4,content_0a91db491d14,13299,review_refresh,stale_but_visible,Moderate confidence in rule match; page qualit...,High impressions may not mean the page needs r...
4,5,content_5feee3994adb,7812,review_refresh,stale_but_visible,Moderate confidence in rule match; page qualit...,High impressions may not mean the page needs r...
5,6,content_c2d929d83eaa,7558,review_refresh,stale_but_visible,Moderate confidence in rule match; page qualit...,High impressions may not mean the page needs r...
6,7,content_b16bd7307b39,4590,review_refresh,stale_but_visible,Moderate confidence in rule match; page qualit...,High impressions may not mean the page needs r...
7,8,content_fe16a55cd13d,4556,review_refresh,stale_but_visible,Moderate confidence in rule match; page qualit...,High impressions may not mean the page needs r...
8,9,content_ecb6215e79fd,4429,review_refresh,stale_but_visible,Moderate confidence in rule match; page qualit...,High impressions may not mean the page needs r...
9,10,content_928af3e22c80,1697,review_refresh,stale_but_visible,Moderate confidence in rule match; page qualit...,High impressions may not mean the page needs r...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks and leakage check

The five lowest-scoring qualifying items have scores of 533, 545, 821, 828, and 954 impressions, respectively. All satisfy the rule's age and visibility thresholds, but the two items with 533 and 545 impressions are close to the visibility cutoff and deserve careful manual review.

A qualifying item may be a weak pick if its page is still accurate, performs its intended purpose well, or has little business value to gain from refreshing. Conversely, an item outside the queue may still need attention for reasons the rule does not capture. The dataset does not provide enough page-level evidence to determine whether these individual pages truly need refreshing.

The score uses only `days_since_last_update` and `impressions_90d`. The audit label derived from `trend_direction`, along with `trend_pct`, is not used as a score input. The code-level leakage check found no overlap between the score inputs and the listed forbidden label fields.

This check supports the claim that the rule does not directly use those label fields. A complete leakage audit must also verify that no product flags or future-window variables enter the scoring logic elsewhere in the notebook. The current check alone does not establish that broader claim.

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Inspect the five lowest-scoring items that qualify for review
weak_picks = queue.loc[queue["score"] > 0].tail(5).copy()

print("Five weakest prioritized picks:")
display(
    weak_picks[
        [
            "rank", "content_id",
            "days_since_last_update", "impressions_90d",
            "score", "reason_code", "action"
        ]
    ]
)

# Check score inputs against known forbidden label fields
forbidden_score_fields = {
    "trend_pct",
    "trend_direction",
    "audit_declining_label",
    "is_declining_label",
}

score_inputs = {
    "days_since_last_update",
    "impressions_90d",
}

print("\nLeakage check:")
print("Score inputs:", sorted(score_inputs))
print("Forbidden score fields:", sorted(forbidden_score_fields))
print(
    "Forbidden fields used as score inputs:",
    sorted(score_inputs & forbidden_score_fields)
)

assert not (score_inputs & forbidden_score_fields), \
    "Potential target leakage detected in score inputs"

# Basic queue integrity checks
assert len(queue) == 30000
assert queue["content_id"].nunique() == len(queue)
assert queue["rank"].is_unique
assert queue["reason_code"].notna().all()
assert queue["action"].notna().all()

print("\nAll queue integrity checks passed.")



Five weakest prioritized picks:


,rank,content_id,days_since_last_update,impressions_90d,score,reason_code,action
12,13,content_7f116ae1f6f5,301,954,954,stale_but_visible,review_refresh
13,14,content_77d4d5930e5e,194,828,828,stale_but_visible,review_refresh
14,15,content_72496874f806,301,821,821,stale_but_visible,review_refresh
15,16,content_6226ee6adc91,183,545,545,stale_but_visible,review_refresh
16,17,content_074ba6ead17b,183,533,533,stale_but_visible,review_refresh



Leakage check:
Score inputs: ['days_since_last_update', 'impressions_90d']
Forbidden score fields: ['audit_declining_label', 'is_declining_label', 'trend_direction', 'trend_pct']
Forbidden fields used as score inputs: []

All queue integrity checks passed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.